# Deep Hedging — Phase 4, brique 1 : simuler Heston

Le modèle de Heston rend la **volatilité elle-même aléatoire**. La variance `v_t` suit un processus de retour à la moyenne (CIR), corrélé au prix :

    dS_t = mu S_t dt + sqrt(v_t) S_t dW1
    dv_t = kappa (theta - v_t) dt + xi sqrt(v_t) dW2
    corr(dW1, dW2) = rho

Paramètres : `kappa` vitesse de retour à la moyenne, `theta` variance long terme, `xi` vol de la vol, `rho` corrélation (négative pour les actions : quand le prix baisse, la vol monte).

Pourquoi ça compte pour le projet : la vol stochastique crée un **risque de vega** que le delta seul ne peut pas couvrir, donc le marché devient **incomplet**. C'est là que le Deep Hedging doit prendre l'avantage.

In [ ]:
import numpy as np
from scipy.stats import norm, kurtosis
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

## Le simulateur (schéma d'Euler à troncature complète)

La variance CIR peut devenir négative sous un schéma d'Euler naïf ; on la tronque à zéro (`full truncation`), le schéma standard et simple. Les deux browniens sont corrélés via `Z2 = rho Z1 + sqrt(1-rho^2) Zperp`.

In [ ]:
def simulate_heston(S0, v0, mu, kappa, theta, xi, rho, T, n, m):
    dt = T/n
    S = np.empty((m, n+1)); v = np.empty((m, n+1))
    S[:, 0] = S0; v[:, 0] = v0
    for k in range(n):
        Z1 = rng.standard_normal(m); Zp = rng.standard_normal(m)
        Z2 = rho*Z1 + np.sqrt(1-rho**2)*Zp          # brownien corrélé à Z1
        vk = np.maximum(v[:, k], 0.0)                # troncature : sqrt d'un v >= 0
        v[:, k+1] = v[:, k] + kappa*(theta - vk)*dt + xi*np.sqrt(vk)*np.sqrt(dt)*Z2
        v[:, k+1] = np.maximum(v[:, k+1], 0.0)       # full truncation
        S[:, k+1] = S[:, k]*np.exp((mu - 0.5*vk)*dt + np.sqrt(vk)*np.sqrt(dt)*Z1)
    return S, v

## Vérifications contre la théorie

In [ ]:
S0, v0, mu = 100., 0.09, 0.05
kappa, theta, xi, rho = 2.0, 0.04, 0.3, -0.7
T, n, m = 1.0, 252, 100_000

S, v = simulate_heston(S0, v0, mu, kappa, theta, xi, rho, T, n, m)
ST, vT = S[:, -1], v[:, -1]

print(f"E[S_T] = {ST.mean():.3f}   theo S0*exp(mu*T) = {S0*np.exp(mu*T):.3f}")
Evt = theta + (v0 - theta)*np.exp(-kappa*T)
print(f"E[v_T] = {vT.mean():.5f}  theo theta+(v0-theta)exp(-kT) = {Evt:.5f}")
logret = np.log(ST/S0)
print(f"kurtosis exces = {kurtosis(logret):.3f}   (0 = normal ; >0 = queues epaisses)")
skew = ((logret - logret.mean())**3).mean() / logret.std()**3
print(f"skewness = {skew:.3f}   (negatif car rho<0 : effet levier)")

## Visualisation : retour à la moyenne de la variance, et queues épaisses

À gauche, quelques trajectoires de variance : elles fluctuent mais reviennent vers `theta`. À droite, la distribution des log-rendements en échelle log : ses queues dépassent celles de la normale ajustée, ce que le GBM ne fait jamais.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
t = np.linspace(0, T, n+1)
for i in range(12):
    a1.plot(t, v[i], lw=0.8)
a1.axhline(theta, color="k", ls="--", lw=1, label="theta (variance long terme)")
a1.set_title("Variance v_t (retour a la moyenne)"); a1.set_xlabel("temps"); a1.set_ylabel("v_t"); a1.legend()

a2.hist(logret, bins=150, density=True, alpha=0.55, color="steelblue", label="Heston")
xg = np.linspace(logret.min(), logret.max(), 400)
a2.plot(xg, norm.pdf(xg, logret.mean(), logret.std()), "r", lw=2, label="normale ajustee")
a2.set_yscale("log"); a2.set_title("Log-rendements : queues epaisses vs normale")
a2.set_xlabel("log(S_T/S0)"); a2.legend()
plt.tight_layout(); plt.show()

## La suite

Brique 2 : le benchmark. On appliquera le **delta de Black-Scholes** (avec une vol constante mal spécifiée) comme couverture, et on verra qu'il laisse un gros risque résiduel de vega. Brique 3 : le couvreur neuronal avec état augmenté (la variance `v_t` observable). On s'attend à un écart bien plus large qu'en GBM.